In [3]:
import pandas as pd
import numpy as np
import torch
import pickle
from typing import Optional, List, Dict, Tuple
from pathlib import Path
import json


In [29]:
def load_medication_data(parquet_path: str) -> pd.DataFrame:
    """
    Load medication data for context tensor creation.

    Attempts parquet first (regardless of file extension), then falls back to
    pandas pickle, raw pickle, and a safe torch.load probe. Normalizes schema
    to include a 'rate/weight_normalized' column by mapping from common
    alternatives when needed.
    """
    print(f"Loading medication data from {parquet_path}...")

    df: Optional[pd.DataFrame] = None
    loaders_tried: List[str] = []

    # Always try parquet first; many .bin files are parquet saved with a custom extension
    try:
        df = pd.read_parquet(parquet_path)
    except Exception as e_parquet:
        loaders_tried.append(f"parquet:{e_parquet}")
        # Try pandas' pickle reader
        try:
            df = pd.read_pickle(parquet_path)
        except Exception as e_pd_pickle:
            loaders_tried.append(f"pd.read_pickle:{e_pd_pickle}")
            # Try raw pickle.load
            try:
                with open(parquet_path, 'rb') as f:
                    obj = pickle.load(f)
                if isinstance(obj, pd.DataFrame):
                    df = obj
                else:
                    raise TypeError("pickled object was not a pandas DataFrame")
            except Exception as e_raw_pickle:
                loaders_tried.append(f"pickle.load:{e_raw_pickle}")
                # Try a lightweight torch.load probe (in case the file was saved via torch)
                try:
                    import torch as _torch
                    obj = _torch.load(parquet_path, map_location="cpu")
                    if isinstance(obj, pd.DataFrame):
                        df = obj
                    elif isinstance(obj, dict) and 'data' in obj and isinstance(obj['data'], pd.DataFrame):
                        df = obj['data']
                    else:
                        raise TypeError("torch file did not contain a pandas DataFrame")
                except Exception as e_torch:
                    loaders_tried.append(f"torch.load:{e_torch}")

    if df is None:
        raise ValueError(
            f"Failed to load medication data from {parquet_path}. Tried: {loaders_tried}"
        )

    # Normalize column names where possible (map common variants to canonical)
    cols_lower_map = {c.lower(): c for c in df.columns}

    def _ensure_column(df_in: pd.DataFrame, canonical: str, candidates: List[str]) -> None:
        for cand in candidates:
            cand_lower = cand.lower()
            if cand_lower in cols_lower_map:
                src = cols_lower_map[cand_lower]
                if src != canonical and canonical not in df_in.columns:
                    df_in.rename(columns={src: canonical}, inplace=True)
                return

    _ensure_column(df, 'hadm_id', ['hadm_id', 'HADM_ID'])
    _ensure_column(df, 'start_time', ['start_time', 'START_TIME', 'STARTTIME'])
    _ensure_column(df, 'end_time', ['end_time', 'END_TIME', 'ENDTIME'])
    _ensure_column(df, 'item_label', ['item_label', 'ITEM_LABEL'])

    # Normalize rate column -> create 'rate/weight_normalized' if missing
    rate_canonical = 'rate/weight_normalized'
    if rate_canonical not in df.columns:
        # Candidates: common variants and fuzzy contains('rate') & contains('weight')
        candidate_order = [
            'rate_weight_normalized',
            'rate_per_kg',
            'rate_per_weight',
            'rate/weight',
            'rate',
            'RATE',
        ]
        picked_src: Optional[str] = None

        for cand in candidate_order:
            if cand in df.columns:
                picked_src = cand
                break
            if cand.lower() in cols_lower_map:
                picked_src = cols_lower_map[cand.lower()]
                break

        if picked_src is None:
            # Fuzzy search: any col name containing both 'rate' and 'weight'
            fuzzy = [c for c in df.columns if ('rate' in c.lower() and 'weight' in c.lower())]
            if len(fuzzy) > 0:
                picked_src = fuzzy[0]

        if picked_src is not None:
            df[rate_canonical] = pd.to_numeric(df[picked_src], errors='coerce')
        else:
            # As a last resort, if a plain 'rate' exists, use it
            if 'rate' in df.columns:
                df[rate_canonical] = pd.to_numeric(df['rate'], errors='coerce')

    # Validate required columns
    required_cols = ['hadm_id', 'start_time', 'end_time', 'item_label', rate_canonical]
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(
            "Missing required medication columns: {}. Available: {}".format(
                missing_cols, list(df.columns)
            )
        )

    # Ensure datetime types and make tz-naive (UTC)
    if not pd.api.types.is_datetime64_any_dtype(df['start_time']):
        df['start_time'] = pd.to_datetime(df['start_time'], errors='coerce')
    if not pd.api.types.is_datetime64_any_dtype(df['end_time']):
        df['end_time'] = pd.to_datetime(df['end_time'], errors='coerce')
    if pd.api.types.is_datetime64tz_dtype(df['start_time']):
        try:
            df['start_time'] = df['start_time'].dt.tz_convert('UTC').dt.tz_localize(None)
        except Exception:
            df['start_time'] = df['start_time'].dt.tz_localize(None)
    if pd.api.types.is_datetime64tz_dtype(df['end_time']):
        try:
            df['end_time'] = df['end_time'].dt.tz_convert('UTC').dt.tz_localize(None)
        except Exception:
            df['end_time'] = df['end_time'].dt.tz_localize(None)

    print(f"Loaded {len(df)} medication events")
    # Clean meds: remove Esmolol; cap each medication at mean + 4*std (per item)
    if 'item_label' in df.columns:
        # Drop Esmolol (case-insensitive)
        mask_esmo = df['item_label'].str.lower() == 'esmolol'
        if mask_esmo.any():
            df = df.loc[~mask_esmo].copy()
            print(f"Removed {int(mask_esmo.sum())} Esmolol rows")

        # Ensure numeric
        df[rate_canonical] = pd.to_numeric(df[rate_canonical], errors='coerce')

        # Compute per-medication caps at mean + 4*std
        stats = df.groupby('item_label')[rate_canonical].agg(['mean', 'std'])
        stats['cap'] = stats['mean'] + 4.0 * stats['std']

        # Merge caps and clip
        df = df.merge(stats['cap'].rename('___cap'), left_on='item_label', right_index=True, how='left')
        before = df[rate_canonical]
        df[rate_canonical] = before.clip(lower=0.0, upper=df['___cap'])
        num_clipped = int((before > df['___cap']).sum())
        df.drop(columns=['___cap'], inplace=True)
        if num_clipped > 0:
            print(f"Capped {num_clipped} medication rows at mean+4*std per item_label")

    return df

In [30]:
medications_df = load_medication_data('/n/netscratch/mzitnik_lab/Lab/rconci/BIOMM/numerics/mv_filtered.bin')

Loading medication data from /n/netscratch/mzitnik_lab/Lab/rconci/BIOMM/numerics/mv_filtered.bin...
Loaded 32410 medication events
Removed 157 Esmolol rows


/n/netscratch/mzitnik_lab/Lab/rconci/tmp/ipykernel_3602124/223475029.py:120: DeprecationWarning: is_datetime64tz_dtype is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.DatetimeTZDtype)` instead.
  if pd.api.types.is_datetime64tz_dtype(df['start_time']):
/n/netscratch/mzitnik_lab/Lab/rconci/tmp/ipykernel_3602124/223475029.py:125: DeprecationWarning: is_datetime64tz_dtype is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.DatetimeTZDtype)` instead.
  if pd.api.types.is_datetime64tz_dtype(df['end_time']):


Capped 107 medication rows at mean+4*std per item_label


In [37]:

def load_waveforms_data(parquet_path: str) -> pd.DataFrame:
    """
    Load waveforms data for context tensor creation.
    """
    print(f"Loading waveforms data from {parquet_path}...")
    df = pd.read_parquet(parquet_path)

    # Ensure absolute_timestamp is datetime and tz-naive (UTC)
    if not pd.api.types.is_datetime64_any_dtype(df['absolute_timestamp']):
        df['absolute_timestamp'] = pd.to_datetime(df['absolute_timestamp'], errors='coerce')
    # If tz-aware, convert to UTC then drop tz
    if pd.api.types.is_datetime64tz_dtype(df['absolute_timestamp']):
        try:
            df['absolute_timestamp'] = df['absolute_timestamp'].dt.tz_convert('UTC').dt.tz_localize(None)
        except Exception:
            # If tz_convert fails (ambiguous/non-existent), just drop tz
            df['absolute_timestamp'] = df['absolute_timestamp'].dt.tz_localize(None)

    # Check for required physio columns
    required_physio = ['ABP MEAN_z', 'CVP_z', 'HR_z', 'RESP_z']
    available_physio = [col for col in required_physio if col in df.columns]
    missing_physio = [col for col in required_physio if col not in df.columns]

    print(f"Available physio measurements: {available_physio}")
    if missing_physio:
        print(f"Missing physio measurements: {missing_physio}")

    return df


In [39]:
waveforms_df = load_waveforms_data('/n/netscratch/mzitnik_lab/Lab/rconci/BIOMM/numerics/smoothed_numerics.bin')

Loading waveforms data from /n/netscratch/mzitnik_lab/Lab/rconci/BIOMM/numerics/smoothed_numerics.bin...
Available physio measurements: ['ABP MEAN_z', 'CVP_z', 'HR_z', 'RESP_z']


/n/netscratch/mzitnik_lab/Lab/rconci/tmp/ipykernel_3602124/1145269912.py:12: DeprecationWarning: is_datetime64tz_dtype is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.DatetimeTZDtype)` instead.
  if pd.api.types.is_datetime64tz_dtype(df['absolute_timestamp']):


In [40]:
waveforms_df.head()

,hadm_id,record_name,absolute_timestamp,ABP MEAN,NBP MEAN,CVP,HR,RESP,record_start_time,record_end_time,icu_admission_time,time_seconds,ABP MEAN_z,CVP_z,HR_z,RESP_z
0,100031,p006892-2140-11-11-22-04n,2140-11-11 22:04:00.056,NaN,NaN,9.60,91.0,12.0,2140-11-11 22:04:00.056,2140-11-15 20:45:00.055999,2140-11-11 20:22:38,0,NaN,0.137797,0.264305,-1.414881
1,100031,p006892-2140-11-11-22-04n,2140-11-11 22:04:10.056,NaN,NaN,9.75,91.0,12.0,2140-11-11 22:04:00.056,2140-11-15 20:45:00.055999,2140-11-11 20:22:38,10,NaN,0.156670,0.264305,-1.414881
2,100031,p006892-2140-11-11-22-04n,2140-11-11 22:04:20.056,NaN,NaN,9.90,91.0,12.0,2140-11-11 22:04:00.056,2140-11-15 20:45:00.055999,2140-11-11 20:22:38,20,NaN,0.175543,0.264305,-1.414881
3,100031,p006892-2140-11-11-22-04n,2140-11-11 22:04:30.056,NaN,NaN,10.05,91.0,12.0,2140-11-11 22:04:00.056,2140-11-15 20:45:00.055999,2140-11-11 20:22:38,30,NaN,0.194416,0.264305,-1.414881
4,100031,p006892-2140-11-11-22-04n,2140-11-11 22:04:40.056,NaN,NaN,10.20,91.0,12.0,2140-11-11 22:04:00.056,2140-11-15 20:45:00.055999,2140-11-11 20:22:38,40,NaN,0.213289,0.264305,-1.414881


In [4]:
df_triggers = medications_df.dropna(subset=['action_cluster_id'])

In [5]:
df_triggers

,subject_id,hadm_id,item_id,input_name,input_class,start_time,end_time,rate,rate_uom,rate/weight,...,prev_rate,trigger,trigger_reason,action_cluster_id,action_cluster_size,action_cluster_rank,first,last,absolute_timestamp,wf_time_delta_s
3,5171,125124,221906,01-Drips,Continuous Med,2171-10-16 08:39:00+00:00,2171-10-16 08:56:00+00:00,0.040033,mcg/kg/min,0.040033,...,0.060038,True,decrease,9.0,2.0,1.0,2171-10-16 07:57:41+00:00,2171-10-20 11:07:11+00:00,2171-10-16 08:39:01+00:00,1.0
4,5171,125124,221906,01-Drips,Continuous Med,2171-10-16 08:56:00+00:00,2171-10-16 09:35:00+00:00,0.080065,mcg/kg/min,0.080065,...,0.040033,True,increase,9.0,2.0,2.0,2171-10-16 07:57:41+00:00,2171-10-20 11:07:11+00:00,2171-10-16 08:56:01+00:00,1.0
10,5171,125124,221906,01-Drips,Continuous Med,2171-10-16 10:44:00+00:00,2171-10-16 11:56:00+00:00,0.080097,mcg/kg/min,0.080097,...,0.060060,True,increase,10.0,1.0,1.0,2171-10-16 07:57:41+00:00,2171-10-20 11:07:11+00:00,2171-10-16 10:44:01+00:00,1.0
12,5171,125124,225828,03-IV Fluid Bolus,Bolus,2171-10-16 11:36:00+00:00,2171-10-16 11:37:00+00:00,NaN,None,NaN,...,1.000000,True,bolus,11.0,1.0,1.0,2171-10-16 07:57:41+00:00,2171-10-20 11:07:11+00:00,2171-10-16 11:36:01+00:00,1.0
14,5171,125124,221906,01-Drips,Continuous Med,2171-10-16 12:20:00+00:00,2171-10-16 12:29:00+00:00,0.100164,mcg/kg/min,0.100164,...,0.070085,True,increase,12.0,3.0,1.0,2171-10-16 07:57:41+00:00,2171-10-20 11:07:11+00:00,2171-10-16 12:20:01+00:00,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32404,99982,151454,221986,01-Drips,Continuous Med,2156-12-04 09:11:00+00:00,2156-12-05 05:56:00+00:00,0.125066,mcg/kg/min,0.125066,...,0.250000,True,decrease,29.0,1.0,1.0,2156-11-28 14:21:06+00:00,2156-12-05 12:09:36+00:00,2156-12-04 09:10:56+00:00,-4.0
32405,99982,151454,220864,04-Fluids (Colloids),Continuous IV,2156-12-04 11:26:00+00:00,2156-12-04 12:26:00+00:00,249.999990,mL/hour,3.511236,...,14.044943,True,decrease,30.0,1.0,1.0,2156-11-28 14:21:06+00:00,2156-12-05 12:09:36+00:00,2156-12-04 11:25:56+00:00,-4.0
32406,99982,151454,220862,04-Fluids (Colloids),Continuous IV,2156-12-04 18:02:00+00:00,2156-12-04 18:32:00+00:00,99.999996,mL/hour,1.404494,...,NaN,True,start,31.0,1.0,1.0,2156-11-28 14:21:06+00:00,2156-12-05 12:09:36+00:00,2156-12-04 18:01:56+00:00,-4.0
32408,99982,151454,221986,01-Drips,Continuous Med,2156-12-05 05:56:00+00:00,2156-12-05 08:14:00+00:00,0.250522,mcg/kg/min,0.250522,...,0.125066,True,increase,32.0,1.0,1.0,2156-11-28 14:21:06+00:00,2156-12-05 12:09:36+00:00,2156-12-05 05:55:56+00:00,-4.0


In [ ]:
patient_meds = 

In [7]:
example_randrop_tensor = torch.load('/n/netscratch/mzitnik_lab/Lab/rconci/BIOMM/processed_data/context_tensors_output/raindrop_context/rd_context_100098_65.pt', weights_only=True)

In [9]:
example_randrop_tensor[0]

tensor([[-0.4268,  0.9579,  0.4720, -0.1265,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0781,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  1.0000,  1.0000,
          1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,
          1.0000,  1.0000,  1.0000,  0.0000,  1.0000,  1.0000,  1.0000,  1.0000,
          1.0000,  1.0000,  1.0000,  1.0000],
        [-0.4441,  0.9454,  0.4615, -0.1470,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0781,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  1.0000,  1.0000,
          1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,
          1.0000,  1.0000,  1.0000,  0.0000,  1.0000,  1.0000, 

In [2]:
import sys, os
import torch

# Allow imports from project modules
sys.path.append('/n/holylfs06/LABS/mzitnik_lab/Lab/rconci/Counterfactual_ICU/src_new/models')
from dataloaders.MIMIC_data import MIMICDataModule

# Configure data roots
data_root = '/n/netscratch/mzitnik_lab/Lab/rconci/BIOMM/processed_data'
icu_stays = '/n/netscratch/mzitnik_lab/Lab/rconci/BIOMM/input_data/ICUSTAYS.csv'

# Build DataModule and fetch one training batch
dm = MIMICDataModule(
    data_root=data_root,
    icu_stays_path=icu_stays,
    batch_size=2,
    num_workers=0,
    random_state=42,
    max_samples=14,
    use_raindrop_context=True,
)
dm.setup('fit')
loader = dm.train_dataloader()

(rd_src, rd_times, rd_length, static_feats, ic_tensor, ic_mask_tensor, p_out_values, p_out_mask, t_Y, med_traj_values, med_traj_mask, med_traj_time_sec, med_tensors) = next(iter(loader))

print('Batch shapes:')
print('  rd_src:', rd_src.shape, 'rd_times:', rd_times.shape, 'rd_length:', rd_length.shape)
print('  static:', static_feats.shape)



[DEBUG] Limited to 14 samples for testing
Split 'train': 14 trajectories
Target trajectory split: 3458/741/742
Actual trajectory split: 3451/735/755
Trajectory percentages: 69.8%/14.9%/15.3%
[DEBUG] Limited to 14 samples for testing
Split 'val': 14 trajectories
Target trajectory split: 3458/741/742
Actual trajectory split: 3451/735/755
Trajectory percentages: 69.8%/14.9%/15.3%
Batch shapes:
  rd_src: torch.Size([2, 6, 52]) rd_times: torch.Size([2, 6]) rd_length: torch.Size([2])
  static: torch.Size([2, 15])


In [7]:
rd_src[0][0, :]

tensor([-1.0968,  0.2812,  0.4187,  0.3111,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0281,  0.0000,  0.0000,  0.0000,  0.0300,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  1.0000,  1.0000,
         0.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,
         1.0000,  1.0000,  1.0000,  0.0000,  1.0000,  1.0000,  1.0000,  0.0000,
         1.0000,  1.0000,  1.0000,  1.0000])

In [8]:
# Build and run Raindrop encoder on the fetched batch
sys.path.append('/n/holylfs06/LABS/mzitnik_lab/Lab/rconci/Counterfactual_ICU/src_new/models')
from raindrop import Raindrop_v2

# Infer d_inp from rd_src last dim (half of it)
d_inp = rd_src.shape[-1] // 2
encoder_hidden_dim = 64
nlayers = 2
model = Raindrop_v2(
    d_inp=d_inp,
    d_model=encoder_hidden_dim,
    nhead=4,
    nhid=128,
    nlayers=nlayers,
    dropout=0.1,
    max_len=rd_src.shape[1],
    d_static=static_feats.shape[-1],
    output_dim=d_inp,  # not used directly in our wrapper
    global_structure=torch.ones(d_inp, d_inp),
    sensor_wise_mask=False,
    static=False,
    debug=False,
)

# Raindrop expects [T,B,2*d_inp], [T,B], lengths [B]
src = rd_src.permute(1,0,2)
times = rd_times.permute(1,0)
lengths = rd_length
with torch.no_grad():
    z, _, _ = model(src=src, static=None, times=times, lengths=lengths)
print('Temporal embedding shape:', z.shape)



/n/holylfs06/LABS/mzitnik_lab/Lab/rconci/micromamba/envs/cf_icu2/lib/python3.10/site-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


[MASK DEBUG] Original mask creation:
  maxlen: 6, batch_size: 2
  lengths: tensor([6, 6])
  lengths shape: torch.Size([2])
[MASK DEBUG] After squeeze:
  Expected shape for transformer: [batch_size, seq_len] = [2, 6]
Temporal embedding shape: torch.Size([2, 68])


In [18]:
z

tensor([[ 0.2616, -1.9387,  0.5636, -0.5242, -0.5311, -0.3206,  0.1042, -0.6179,
         -0.2879,  0.9790,  1.2504,  0.1429, -1.0017,  0.7210, -0.0878,  0.8264,
         -0.3748, -0.4846, -0.1779,  0.0812, -0.4391,  0.7140,  0.3439, -0.1860,
         -0.5005,  0.2368, -0.5723, -0.6046,  0.2601, -0.5452,  0.4321,  0.4441,
         -0.1865, -0.1385,  0.2548, -0.5720, -0.5431,  0.2318,  0.9205, -0.2548,
         -1.4310,  0.6822,  1.8894,  0.2735, -0.3735, -0.1954,  1.1234,  0.3565,
          0.1673, -0.3215, -1.0311, -1.1548, -1.6030, -1.0151, -1.1822, -0.9840,
          0.0262, -0.8600, -0.0873, -1.5011,  0.9361,  0.8586,  0.8997,  1.6790,
          0.6594,  1.4611,  1.0278,  1.8212],
        [ 0.3008, -2.0333,  0.5655, -0.5209, -0.5649, -0.2771,  0.4614, -0.7312,
         -0.2296,  0.8950,  1.1547,  0.1077, -1.0477,  0.8220, -0.0217,  0.7681,
         -0.3263, -0.3549, -0.2177,  0.0657, -0.3448,  0.7401,  0.3302, -0.2832,
         -0.6367,  0.2162, -0.5668, -0.6943,  0.1889, -0.4540, 